In [1]:
from os import path, listdir, mkdir, remove
import shutil
from time import sleep

In [2]:
src_path = r'/mnt/c/Users/sarth/Downloads/Test_folder'
replica_path = r'/mnt/c/Users/sarth/Downloads/Replica_folder'

In [3]:
# def copy_directory_with_content(src:str, replica:str):
#     '''This only copies directory from src to replica. Does not delete files in replica. Does not check for last modified.'''
#     element_ls = listdir(src)
#     # print(f'Elements: {listdir(src)}') # diag

#     for element in element_ls: # for files only
#         if path.isfile(path.join(src, element)):
#             if not path.exists(path.join(replica, element)):
#                 print(f'Copying {path.join(replica, element)}')
#                 shutil.copyfile(path.join(src, element), path.join(replica, element))
#         else: # for directories only
#             if not path.exists(path.join(replica, element)):
#                 print(f'Copying entire directory to: {path.join(replica, element)}')
#                 shutil.copytree(path.join(src, element), path.join(replica, element))
#             else:        # if the directory exists we need to traverse it and copy missing elements
#                 copy_directory_with_content(path.join(src, element), path.join(replica, element))
#                 # print('Element:', element) # diag

In [4]:
# if not path.exists(replica_path):
#     print(f'Creating replica directory')
#     mkdir(replica_path)

# copy_directory_with_content(src_path, replica_path)

In [5]:
# def copy_directory_with_modified_content(src:str, replica:str):
#     '''This only copies directory from src to replica. Also copies changes in files. Does not delete files in replica.'''
#     element_ls = listdir(src)
#     # print(f'Elements: {listdir(src)}') # diag

#     for element in element_ls: # for files only
#         if path.isfile(path.join(src, element)):
#             if not path.exists(path.join(replica, element)):
#                 print(f'Copying {path.join(replica, element)}')
#                 shutil.copyfile(path.join(src, element), path.join(replica, element))
#             elif path.getmtime(path.join(src, element)) > path.getmtime(path.join(replica, element)):  # check for last modified between replica and src
#                 print(f'Copying {path.join(replica, element)}')
#                 shutil.copyfile(path.join(src, element), path.join(replica, element))
                
#         else: # for directories only
#             if not path.exists(path.join(replica, element)):
#                 print(f'Copying entire directory: {path.join(replica, element)}')
#                 shutil.copytree(path.join(src, element), path.join(replica, element))
#             else:        # if the directory exists we need to traverse it and copy missing elements
#                 copy_directory_with_modified_content(path.join(src, element), path.join(replica, element))
#                 # print('Element:', element) # diag

In [6]:
# if not path.exists(replica_path):
#     print(f'Creating replica directory')
#     mkdir(replica_path)

# copy_directory_with_modified_content(src_path, replica_path)

In [7]:
def sync_dir_changes(src:str, replica:str):
    '''This only copies directory from src to replica. Also copies changes in files. Delete extra files found in replica.'''
    for element in listdir(src):
        if path.isfile(path.join(src, element)): # for files only
            if not path.exists(path.join(replica, element)):
                print(f'Copying {path.join(replica, element)}')
                shutil.copyfile(path.join(src, element), path.join(replica, element))
            elif path.getmtime(path.join(src, element)) > path.getmtime(path.join(replica, element)):  # compare for modification time between replica and src
                print(f'Copying {path.join(replica, element)}')
                shutil.copyfile(path.join(src, element), path.join(replica, element))
                
        else: # for directories only
            if not path.exists(path.join(replica, element)):
                print(f'Copying entire directory: {path.join(replica, element)}')
                shutil.copytree(path.join(src, element), path.join(replica, element))
            else:        # if the directory exists we need to traverse it and copy missing elements
                for replica_element in listdir(path.join(replica, element)):
                    if not path.exists(path.join(src, element, replica_element)):
                        replica_element_path = path.join(replica, element, replica_element)
                        if path.isfile(replica_element_path):
                            print(f'Removing {replica_element_path}')
                            remove(replica_element_path) # delete a single file
                        else:
                            print(f'Removing entire directory: {replica_element_path}')
                            shutil.rmtree(replica_element_path) # delete dir tree

                sync_dir_changes(path.join(src, element), path.join(replica, element)) # recursion to propagate the sync down the directory tree

def create_base_sync_changes(src:str, replica:str):
    if not path.exists(replica):
        print(f'Creating replica directory: {replica}')
        mkdir(replica)
    else: # deletes extra dir from the base folder
        for replica_element in listdir(replica):
            if not path.exists(path.join(src, replica_element)):
                replica_element_path = path.join(replica, replica_element)
                if path.isfile(replica_element_path):
                    print(f'Removing {replica_element_path}')
                    remove(replica_element_path) # delete a single file
                else:
                    print(f'Removing entire directory: {replica_element_path}')
                    shutil.rmtree(replica_element_path) # delete dir tree
    sync_dir_changes(src, replica)

In [8]:
create_base_sync_changes(src_path, replica_path)

In [ ]:
sync_attempts = 0
while sync_attempts < 360:
    create_base_sync_changes(src_path, replica_path)
    sleep(10) # sync interval
    sync_attempts += 1

Removing entire directory: /mnt/c/Users/sarth/Downloads/Replica_folder/subfolder 3


In [10]:
# Question: should we traverse a directory if it has not been modified? (efficiency)